In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import transformers
from transformers import CamembertForSequenceClassification, CamembertTokenizer, Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
import datasets
from datasets import Dataset
import sklearn
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# CamemBERT fine-tuné

In [2]:
df_avis = pd.read_csv('./../data/labelled_topics/dataset_avis.csv')

In [3]:
TEXT_COL = "clean_comment"
LABEL_COLS = ["qualité produit", "service livraison", "service client"]

label_names = LABEL_COLS
num_labels = len(label_names)

print(label_names)

['qualité produit', 'service livraison', 'service client']


In [4]:
X = df_avis[TEXT_COL].values
y = df_avis[LABEL_COLS].values

texts, X_test, labels, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
df_avis_gold = pd.read_csv('./../data/test_dataset/100_avis_annote.csv', sep=";")
X_gold = df_avis_gold[TEXT_COL].values
y_gold = df_avis_gold[LABEL_COLS].values

## Entraînement avec cross validation

#### Sélection des hyperparamètres

In [7]:
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [9]:
all_fold_metrics = []

for fold, (train_index, val_index) in enumerate(kf.split(texts)):
    print(f"\nFold {fold + 1}/{k}")

    # Split train/val
    train_texts, val_texts = texts[train_index], texts[val_index]
    train_labels, val_labels = labels[train_index], labels[val_index]

    # Tokenizer
    tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
    
    # Créer des datasets Hugging Face
    def tokenize_and_format(texts, labels):
        encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=256)
        encodings['labels'] = labels.astype(np.float32)
        return encodings

    train_enc = tokenize_and_format(train_texts, train_labels)
    val_enc = tokenize_and_format(val_texts, val_labels)

    train_ds = Dataset.from_dict(train_enc)
    val_ds = Dataset.from_dict(val_enc)

    # Modèle
    model = CamembertForSequenceClassification.from_pretrained(
        "camembert-base",
        num_labels=len(LABEL_COLS),
        problem_type="multi_label_classification"
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Trainer
    training_args = TrainingArguments(
        output_dir=f"./../models/classification/camembert/camembert_fold{fold}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_micro",
        fp16=True  # accélération GPU
    )

    # Fonction métriques
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.sigmoid(torch.tensor(logits))
        preds = (probs > 0.5).int().numpy()
        labels = labels.astype(int)
        f1_micro = f1_score(labels, preds, average="micro")
        return {"f1_micro": f1_micro}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Entraînement
    trainer.train()

    # Évaluation finale sur ce fold
    metrics = trainer.evaluate()
    print(metrics)
    all_fold_metrics.append(metrics['eval_f1_micro'])

# Moyenne sur les folds
print("F1 micro moyen sur les folds:", np.mean(all_fold_metrics))


Fold 1/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_10125/4146794912.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro
1,0.590200,0.555422,0.543452
2,0.565500,0.534009,0.589327
3,0.553200,0.526274,0.593897


{'eval_loss': 0.5262739658355713, 'eval_f1_micro': 0.5938967136150235, 'eval_runtime': 1.9127, 'eval_samples_per_second': 228.476, 'eval_steps_per_second': 14.639, 'epoch': 3.0}

Fold 2/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_10125/4146794912.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Epoch,Training Loss,Validation Loss,F1 Micro
1,0.589800,0.571230,0.456410
2,0.555700,0.555672,0.516432
3,0.547000,0.551253,0.522248


{'eval_loss': 0.5512527823448181, 'eval_f1_micro': 0.522248243559719, 'eval_runtime': 1.9962, 'eval_samples_per_second': 218.413, 'eval_steps_per_second': 14.027, 'epoch': 3.0}

Fold 5/5


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_10125/4146794912.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro
1,0.584700,0.587138,0.534464
2,0.560000,0.572796,0.525714
3,0.560400,0.566883,0.535433


{'eval_loss': 0.5668833255767822, 'eval_f1_micro': 0.5354330708661418, 'eval_runtime': 1.9633, 'eval_samples_per_second': 222.077, 'eval_steps_per_second': 14.262, 'epoch': 3.0}
F1 micro moyen sur les folds: 0.5510957278349801


#### Entraînement final

In [25]:
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

def tokenize_and_format(texts, labels):
        encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=256)
        encodings['labels'] = labels.astype(np.float32)
        return encodings

train_enc = tokenize_and_format(texts, labels)

In [26]:
train_ds = Dataset.from_dict(train_enc)

In [27]:
model_final = CamembertForSequenceClassification.from_pretrained(
    "camembert-base",
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_final.to(device)

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CamembertForSequenceClassification(
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [28]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()  # ajuster les seuils optimaux
    labels = labels.astype(int)
    f1_micro = f1_score(labels, preds, average="micro")
    return {"f1_micro": f1_micro}

In [29]:
training_args_final = TrainingArguments(
    output_dir="../models/classification/camembert/camembert_crossvalidation",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2, 
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    fp16=True
)

In [30]:
trainer_final = Trainer(
    model=model_final,
    args=training_args_final,
    train_dataset=train_ds,
    compute_metrics=compute_metrics
)

In [31]:
trainer_final.train()

Step,Training Loss
100,0.612300
200,0.563800


TrainOutput(global_step=274, training_loss=0.5790641255622363, metrics={'train_runtime': 63.139, 'train_samples_per_second': 69.086, 'train_steps_per_second': 4.34, 'total_flos': 573850364064768.0, 'train_loss': 0.5790641255622363, 'epoch': 2.0})

In [32]:
trainer_final.save_model("../models/classification/camembert/camembert_crossvalidation")
tokenizer.save_pretrained("../models/classification/camembert/camembert_crossvalidation")

('./../models/classification/camembert/camembert_final/tokenizer_config.json',
 './../models/classification/camembert/camembert_final/special_tokens_map.json',
 './../models/classification/camembert/camembert_final/sentencepiece.bpe.model',
 './../models/classification/camembert/camembert_final/added_tokens.json')

#### Evaluation sur le dataset de test

In [33]:
# Tokenisation
test_enc = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)
test_enc['labels'] = torch.tensor(y_test, dtype=torch.float)

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: v[idx] for k,v in self.encodings.items()}

test_ds = TestDataset(test_enc)

# Evaluation
trainer_final.eval_dataset = test_ds
metrics = trainer_final.evaluate()
print(metrics)

{'eval_loss': 0.540988564491272, 'eval_f1_micro': 0.5837145471180238, 'eval_runtime': 2.1601, 'eval_samples_per_second': 252.767, 'eval_steps_per_second': 16.203, 'epoch': 2.0}


In [34]:
texts = list(X_test)
labels_true = torch.tensor(y_test, dtype=torch.int)

# Préparer les inputs
inputs = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
inputs = {k: v.to(model_final.device) for k,v in inputs.items()}

# Prédictions
with torch.no_grad():
    logits = model_final(**inputs).logits
probs = torch.sigmoid(logits).cpu()

# Appliquer les seuils pour chaque label
thresholds = [0.5, 0.5, 0.5]
preds = (probs > torch.tensor(thresholds)).int()

# F1 score
f1_micro = f1_score(labels_true, preds, average='micro')
f1_macro = f1_score(labels_true, preds, average='macro')

print("F1 micro:", f1_micro)
print("F1 macro:", f1_macro)

F1 micro: 0.5837145471180238
F1 macro: 0.41899711718536287


In [35]:
print(classification_report(labels_true, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.67      0.62      0.64       226
service livraison       0.60      0.63      0.62       284
   service client       0.00      0.00      0.00        75

        micro avg       0.63      0.55      0.58       585
        macro avg       0.42      0.42      0.42       585
     weighted avg       0.55      0.55      0.55       585
      samples avg       0.58      0.55      0.56       585



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [36]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(labels_true[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,251,69
Vrai 1,87,139


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,142,120
Vrai 1,104,180


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,471,0
Vrai 1,75,0


#### Evaluation sur le dataset gold annoté manuellement

In [37]:
# Tokenisation
gold_enc = tokenizer(
    list(X_gold),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)
gold_enc['labels'] = torch.tensor(y_gold, dtype=torch.float)

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: v[idx] for k,v in self.encodings.items()}

gold_ds = TestDataset(gold_enc)

# Evaluation
trainer_final.eval_dataset = gold_ds
metrics = trainer_final.evaluate()
print(metrics)

{'eval_loss': 0.7555344104766846, 'eval_f1_micro': 0.20809248554913296, 'eval_runtime': 0.2133, 'eval_samples_per_second': 468.775, 'eval_steps_per_second': 32.814, 'epoch': 2.0}


In [38]:
texts = list(X_gold)
labels_true = torch.tensor(y_gold, dtype=torch.int)

# Préparer les inputs
inputs = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
inputs = {k: v.to(model_final.device) for k,v in inputs.items()}

# Prédictions
with torch.no_grad():
    logits = model_final(**inputs).logits
probs = torch.sigmoid(logits).cpu()

# Appliquer les seuils pour chaque label
thresholds = [0.5, 0.5, 0.5]
preds = (probs > torch.tensor(thresholds)).int()

In [39]:
print(classification_report(labels_true, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.20      0.07      0.11        41
service livraison       0.22      0.71      0.33        21
   service client       0.00      0.00      0.00        27

        micro avg       0.21      0.20      0.21        89
        macro avg       0.14      0.26      0.15        89
     weighted avg       0.14      0.20      0.13        89
      samples avg       0.18      0.14      0.15        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill

In [40]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(labels_true[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,47,12
Vrai 1,38,3


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,25,54
Vrai 1,6,15


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,73,0
Vrai 1,27,0


## Entraînement avec dataset de validation

#### Entraînement

In [6]:
dataset = Dataset.from_pandas(
    df_avis[[TEXT_COL] + LABEL_COLS],
    preserve_index=True
)

dataset = dataset.train_test_split(test_size=0.20, seed=42)
train_ds = dataset["train"]
test_ds = dataset["test"]

dataset = train_ds.train_test_split(test_size=0.15, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

In [7]:
model_name = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1853 [00:00<?, ? examples/s]

Map:   0%|          | 0/328 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

In [8]:
def pack_labels(batch):
    batch["labels"] = np.stack([batch[col] for col in LABEL_COLS], axis=1).astype(np.float32)
    return batch

train_ds = train_ds.map(pack_labels, batched=True)
test_ds = test_ds.map(pack_labels, batched=True)
val_ds = val_ds.map(pack_labels, batched=True)

Map:   0%|          | 0/1853 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

Map:   0%|          | 0/328 [00:00<?, ? examples/s]

In [9]:
cols_to_remove = LABEL_COLS + [TEXT_COL]
train_ds = train_ds.remove_columns(cols_to_remove)
val_ds = val_ds.remove_columns(cols_to_remove)

# on garde le texte pour l'analyse des erreurs
test_ds_for_errors = test_ds
test_ds = test_ds.remove_columns(LABEL_COLS + [TEXT_COL])

train_ds.set_format("torch")
test_ds.set_format("torch")
val_ds.set_format("torch")

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CamembertForSequenceClassification(
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [13]:
training_args = TrainingArguments(
    output_dir="../models/classification/camembert/camembert_valdataset",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_2430/1073348029.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.606200,0.572481,0.553797,0.404638
2,0.568200,0.553300,0.574046,0.418441
3,0.536800,0.539265,0.597015,0.439103


TrainOutput(global_step=348, training_loss=0.5637049400943449, metrics={'train_runtime': 222.9853, 'train_samples_per_second': 24.93, 'train_steps_per_second': 1.561, 'total_flos': 731323744574976.0, 'train_loss': 0.5637049400943449, 'epoch': 3.0})

In [16]:
trainer.save_model("./../models/classification/camembert/camembert_valdataset")
tokenizer.save_pretrained("./../models/classification/camembert/camembert_valdataset")

('./../models/classification/camembert/camembert_valdataset/tokenizer_config.json',
 './../models/classification/camembert/camembert_valdataset/special_tokens_map.json',
 './../models/classification/camembert/camembert_valdataset/tokenizer.json')

#### Prédiction

In [17]:
def predict_texts(texts, threshold=0.5):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # déplacer les inputs sur le même device que le modèle
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits)
    preds = (probs > threshold).int()

    return probs.cpu().numpy(), preds.cpu().numpy()


In [18]:
texts_test = ["Livraison lente mais service client correct"]

probs, preds = predict_texts(texts_test)

for i, label in enumerate(label_names):
    print(label, probs[0][i], preds[0][i])

qualité produit 0.15563639 0
service livraison 0.63199085 1
service client 0.2950499 0


#### Trouver le bon seuil

In [19]:
# Mettre le modèle en mode évaluation
model.eval()

val_labels = []
val_probs = []

device = next(model.parameters()).device

for batch in val_ds:
    input_ids = batch["input_ids"].unsqueeze(0).to(device)
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    val_labels.append(labels.cpu().numpy())
    val_probs.append(probs.cpu().numpy())

# concaténer toutes les batches
val_labels = np.concatenate(val_labels, axis=0)
val_probs = np.concatenate(val_probs, axis=0)

print(val_labels.shape, val_probs.shape)

(328, 3) (328, 3)


In [20]:
num_labels = val_labels.shape[1]
best_thresholds = []

for i in range(num_labels):
    best_f1 = 0
    best_t = 0.0
    for t in np.arange(0.1, 0.91, 0.01):
        y_pred = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_labels[:, i], y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    best_thresholds.append(best_t)

print("Meilleurs seuils par label :", best_thresholds)

Meilleurs seuils par label : [0.2599999999999999, 0.43999999999999984, 0.22999999999999995]


#### Evaluation sur le dataset de test

In [21]:
# Mettre le modèle en mode évaluation
model.eval()

test_labels = []
test_probs = []

device = next(model.parameters()).device

for batch in test_ds:
    # déplacer les tenseurs sur le même device que le modèle
    input_ids = batch["input_ids"].unsqueeze(0).to(device)  # ajouter batch dimension si nécessaire
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    test_labels.append(labels.cpu().numpy())
    test_probs.append(probs.cpu().numpy())

# concaténer tous les batches
test_labels = np.concatenate(test_labels, axis=0)
test_probs = np.concatenate(test_probs, axis=0)

print(test_labels.shape, test_probs.shape)

(546, 3) (546, 3)


In [22]:
preds = np.zeros_like(test_probs, dtype=int)
for i, t in enumerate(best_thresholds):
    preds[:, i] = (test_probs[:, i] > t).astype(int)

In [23]:
print(classification_report(test_labels, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.55      0.84      0.67       203
service livraison       0.59      0.88      0.70       294
   service client       0.34      0.76      0.47        82

        micro avg       0.53      0.85      0.65       579
        macro avg       0.49      0.83      0.61       579
     weighted avg       0.54      0.85      0.66       579
      samples avg       0.55      0.85      0.65       579



In [24]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(test_labels[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,205,138
Vrai 1,33,170


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,68,184
Vrai 1,34,260


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,342,122
Vrai 1,20,62


#### Evaluation sur le dataset gold annoté manuellement

In [25]:
gold_ds = Dataset.from_pandas(
    df_avis_gold[[TEXT_COL] + LABEL_COLS],
    preserve_index=True
)

gold_ds = gold_ds.map(tokenize, batched=True)
gold_ds = gold_ds.map(pack_labels, batched=True)

# on garde le texte pour l'analyse des erreurs
gold_ds_for_errors = test_ds
gold_ds = gold_ds.remove_columns(LABEL_COLS + [TEXT_COL])
gold_ds.set_format("torch")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [26]:
# Mettre le modèle en mode évaluation
model.eval()

gold_labels = []
gold_probs = []

device = next(model.parameters()).device

for batch in gold_ds:
    # déplacer les tenseurs sur le même device que le modèle
    input_ids = batch["input_ids"].unsqueeze(0).to(device)  # ajouter batch dimension si nécessaire
    attention_mask = batch["attention_mask"].unsqueeze(0).to(device)
    labels = batch["labels"].unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    probs = torch.sigmoid(logits)

    gold_labels.append(labels.cpu().numpy())
    gold_probs.append(probs.cpu().numpy())

# concaténer tous les batches
gold_labels = np.concatenate(gold_labels, axis=0)
gold_probs = np.concatenate(gold_probs, axis=0)

print(gold_labels.shape, gold_probs.shape)

(100, 3) (100, 3)


In [27]:
preds = np.zeros_like(gold_probs, dtype=int)
for i, t in enumerate(best_thresholds):
    preds[:, i] = (gold_probs[:, i] > t).astype(int)

In [28]:
print(classification_report(gold_labels, preds, target_names=LABEL_COLS))

                   precision    recall  f1-score   support

  qualité produit       0.37      0.44      0.40        41
service livraison       0.20      0.86      0.32        21
   service client       0.04      0.04      0.04        27

        micro avg       0.22      0.42      0.29        89
        macro avg       0.20      0.44      0.25        89
     weighted avg       0.23      0.42      0.27        89
      samples avg       0.21      0.28      0.23        89



/mnt/c/Users/charb/Documents/ALTERNANCE/Datascientest/Projet/TrustPilot/DS/nov24_alt_trustpilot/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [29]:
for i, label in enumerate(LABEL_COLS):
    cm = confusion_matrix(gold_labels[:, i], preds[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"Matrice de confusion — {label}")
    display(cm_df)

Matrice de confusion — qualité produit


,Prédit 0,Prédit 1
Vrai 0,28,31
Vrai 1,23,18


Matrice de confusion — service livraison


,Prédit 0,Prédit 1
Vrai 0,6,73
Vrai 1,3,18


Matrice de confusion — service client


,Prédit 0,Prédit 1
Vrai 0,49,24
Vrai 1,26,1
